# NLU HOSTAGE — ai_1_nlu_v1

Notebook ini hanya memakai modul training bersama. Semua cell legacy telah dihapus.


In [1]:
from pathlib import Path
DATASET_FILENAME = "v1_chat_dataset_100.csv"
NOTEBOOK_FOLDER = "ai_1_nlu_v1"
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != NOTEBOOK_FOLDER:
    candidate = NOTEBOOK_DIR / NOTEBOOK_FOLDER
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate
DATASET_PATH = NOTEBOOK_DIR / "data" / DATASET_FILENAME
print(f"Dataset aktif: {DATASET_PATH}")


Dataset aktif: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v1\data\v1_chat_dataset_100.csv


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(NOTEBOOK_DIR).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.nlu_eda import run_nlu_eda
from modules.nlu_training import (
    predict_intent as _predict_intent_shared,
    predict_transformer_intent as _predict_transformer_intent_shared,
    run_all_nlu_models,
    train_naive_bayes,
    train_naive_bayes_tuned,
    train_svm,
    train_svm_tuned,
    train_transformer,
)

MODEL_DIR = Path(NOTEBOOK_DIR) / "models"
# Kebijakan proyek: SVM/NB pada CPU, Transformer pada GPU CUDA.
CLASSICAL_DEVICE = "cpu"
TRANSFORMER_DEVICE = "cuda"
# Fine-tuning: seluruh bobot IndoBERT dilatih. Epoch adalah batas maksimum.
TRANSFORMER_EPOCHS = 12
TRANSFORMER_OPTIONS = {
    "learning_rates": (1e-5, 2e-5, 3e-5),  # pilih hanya dari validation macro-F1
    "validation_size": 0.15,  # train/validation/test sekitar 65/15/20
    "batch_size": 8,
    "eval_batch_size": 16,
    "gradient_accumulation_steps": 2,  # effective batch 16 pada satu GPU
    "max_length": 128,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "linear",
    "dropout": 0.1,
    "label_smoothing_factor": 0.0,
    "max_grad_norm": 1.0,
    "early_stopping_patience": 3,
    "early_stopping_threshold": 0.001,
    "gradient_checkpointing": False,  # aktifkan jika memori GPU tidak cukup
}


def run_eda(plot=True):
    return run_nlu_eda(DATASET_PATH, plot=plot)

def train_svm_model():
    return train_svm(DATASET_PATH, MODEL_DIR)

def train_svm_tuned_model():
    return train_svm_tuned(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_model():
    return train_naive_bayes(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_tuned_model():
    return train_naive_bayes_tuned(DATASET_PATH, MODEL_DIR)

def train_transformer_model(epochs=None, **overrides):
    options = {**TRANSFORMER_OPTIONS, **overrides}
    # Untuk satu LR: train_transformer_model(learning_rates=None, learning_rate=2e-5)
    return train_transformer(
        DATASET_PATH, MODEL_DIR,
        epochs=TRANSFORMER_EPOCHS if epochs is None else epochs,
        device=TRANSFORMER_DEVICE, **options,
    )

def predict_intent(text, model_filename=None):
    return _predict_intent_shared(text, MODEL_DIR, model_filename)

def predict_transformer_intent(text):
    return _predict_transformer_intent_shared(text, MODEL_DIR, device=TRANSFORMER_DEVICE)

HOSTAGE_TEST_CASES = [
    ("accusing", "B kena Gag Order saat menjelaskan alibi, menurut gw itu pola Hitman."),
    ("defending", "Gw bukan Hitman, tuduhan itu gak punya bukti publik."),
    ("bluffing", "Gw Spy, semalam gw Guard Raka dan dia pasti aman."),
    ("probing", "Stalker, semalam lu Peek siapa dan hasilnya apa?"),
    ("deflecting", "Jangan fokus ke gw, cek D yang terus mengubah cerita tiap ditanya."),
    ("persuading", "Vote C aja, dia paling diuntungkan dari korban Hostage semalam."),
    ("claiming", "Klaim gw Civilian, gw gak punya skill malam."),
    ("neutral", "Fase malam bikin chat terkunci, kita tunggu pagi dulu."),
]

def run_hostage_test_suite(model_filename=None):
    correct = 0
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = predict_intent(chat, model_filename)
        correct += predicted == expected
        print(f"{expected:12} | prediksi={predicted:12} | confidence={confidence:6.2f}% | {chat}")
    print(f"\nCocok: {correct}/{len(HOSTAGE_TEST_CASES)}")

print("Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),")
print("train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().")
print("Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.")
print("Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.")


In [ ]:
# Jalankan baseline CPU dan fine-tuning IndoBERT CUDA; tuning SVM/NB opsional.
RUN_TRANSFORMER = True
RUN_TUNING = False  # khusus grid search SVM/NB; IndoBERT memakai TRANSFORMER_OPTIONS
artifacts, hasil_training, hasil_manual_test = run_all_nlu_models(
    DATASET_PATH, MODEL_DIR,
    run_transformer=RUN_TRANSFORMER,
    run_tuning=RUN_TUNING,
    transformer_epochs=TRANSFORMER_EPOCHS,
    transformer_device=TRANSFORMER_DEVICE,
    transformer_options=TRANSFORMER_OPTIONS,
)
print('RINGKASAN EVALUASI HOLDOUT:')
display(hasil_training)
print('RINGKASAN 10 CHAT UJI:')
display(hasil_manual_test)

# Riwayat loss dan validation macro-F1 per epoch; test tidak dipakai memilih model.
if "IndoBERT Transformer" in artifacts:
    import pandas as pd
    transformer_summary = artifacts["IndoBERT Transformer"]["artifact"]["training_summary"]
    display(pd.DataFrame([
        {k: v for k, v in trial.items() if k not in {"history", "best_checkpoint"}}
        for trial in transformer_summary["trials"]
    ]))
    for trial in transformer_summary["trials"]:
        print(f"Learning rate: {trial['learning_rate']:g}")
        display(pd.DataFrame(trial["history"]))


## Catatan evaluasi

Konfigurasi sekarang memakai train/validation/test sekitar 65/15/20. Test 20% tetap
sama dengan SVM/NB; validation diambil dari pool train IndoBERT. Setiap trial mulai
dari pretrained, lalu checkpoint dan learning rate dipilih berdasarkan validation
macro-F1. Test diprediksi sekali setelah seluruh trial selesai.

Maksimum 12 epoch, early stopping patience 3 dengan ambang perbaikan 0,001.
Ini batas eksperimen, bukan jaminan bahwa 12 epoch lebih baik. Ubah parameter di
`TRANSFORMER_OPTIONS`. Untuk hanya melatih IndoBERT, jalankan
`hasil_transformer = train_transformer_model()` setelah cell setup.

Skor eksekusi lama dihapus dari cell yang berubah karena sebelumnya test dipakai
untuk seleksi checkpoint. Jalankan ulang untuk mendapatkan hasil protokol baru.
Ringkasan konfigurasi, loss per epoch, confusion matrix, dan prediksi test tersimpan
di folder model. Uji 10 chat hanya pemeriksaan contoh, bukan estimasi generalisasi.
